# Long-Context LLM Pipeline — FinanceBench (full batch run)

**Thesis:** Vector RAG vs. Vectorless RAG vs. Long-Context LLMs on FinanceBench

---

## What this pipeline is

No retrieval, no chunking, no embeddings. The entire PDF is placed directly in
the generation model's context window and the model answers from that alone.
Feasible here because both generation models (Gemini 3.5 Flash, DeepSeek V4)
have 1M-token context windows — see CLAUDE.md.

This notebook follows the same shape as `vector_rag_pipeline.ipynb`: each stage
is resumable and saves its own output file before the next stage starts, so a
partial run can always be picked up where it left off. It runs fully locally —
no Colab/Drive mount needed, since nothing here requires a GPU (there's no
embedding model to load, unlike vector RAG's Stella).

Same reason vector_rag_pipeline.ipynb gives for inlining everything rather
than importing `evaluation/*.py`: keeping the three pipelines self-contained
makes them easier to keep independent and directly comparable. `evaluation/*.py`
still exists on disk with the same logic; this notebook doesn't import it.

## Pipeline stages

```
③ Parse      — PDF -> full document text (page-bounded, no chunking)
④ Generate   — entire document text + question -> answer
⑤ Score      — Correct / Incorrect / Failure to Answer
⑥ Summarize  — answer quality + token/cost totals
⑦ Latency    — dedicated sequential timing pass, median-of-N
```

Retrieval quality (Recall@k / MRR@k) does not apply to this pipeline — there's
no retrieval step to evaluate. Per CLAUDE.md, this is acknowledged in the
thesis, not treated as missing data.

---
## Stage 0 — Setup

Local paths, `.env`, cost tracking. No Drive mount, no git clone — we're already
inside the repo.

In [ ]:
import os, sys, json, time, re, textwrap
from pathlib import Path
from dataclasses import asdict, dataclass
from functools import wraps

import pandas as pd
from dotenv import load_dotenv

# Walk up from the notebook's cwd until we find the repo root (has pyproject.toml).
# Works regardless of whether Jupyter was launched from the repo root or from
# pipelines/long_context/.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Could not find repo root (no pyproject.toml found walking up from cwd)")

DATA_DIR = REPO_ROOT / "data"
PDF_DIR  = REPO_ROOT / "pdfs"

RESULTS_DIR     = REPO_ROOT / "experiments" / "results"
DOCS_DIR        = RESULTS_DIR / "long_context_docs"                 # one parsed-text file per document
GENERATION_PATH = RESULTS_DIR / "long_context_stage_generation.jsonl"
SCORING_PATH    = RESULTS_DIR / "long_context_stage_scoring.jsonl"
COSTS_PATH      = RESULTS_DIR / "long_context_costs.jsonl"
LATENCY_PATH    = RESULTS_DIR / "long_context_latency.jsonl"

# override=True: without it, re-running this cell after editing .env keeps the
# stale value already sitting in the kernel's os.environ from an earlier run
# in this session, instead of picking up your edit.
load_dotenv(REPO_ROOT / ".env", override=True)
GOOGLE_API_KEY   = os.getenv("GOOGLE_API_KEY", "")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "")
GROQ_API_KEY     = os.getenv("GROQ_API_KEY", "")

print(f"REPO_ROOT: {REPO_ROOT}")
for name, val in [
    ("GOOGLE_API_KEY", GOOGLE_API_KEY),
    ("DEEPSEEK_API_KEY", DEEPSEEK_API_KEY),
    ("GROQ_API_KEY", GROQ_API_KEY),
]:
    print(f"  {name:<18}: {'ok' if val else 'MISSING - fill in .env'}")

In [ ]:
# --- Cost tracking (same logic as evaluation/cost_tracker.py, inlined --
# see the "What this pipeline is" note above for why) ---

PRICING_PER_MILLION_TOKENS: dict = {
    "gemini-3.5-flash": {"input": None, "output": None},
    "deepseek-v4": {"input": None, "output": None},
    "openai/gpt-oss-120b": {"input": None, "output": None},  # judge, via Groq (free tier)
}


@dataclass
class UsageRecord:
    timestamp: float
    pipeline: str
    stage: str
    model: str
    doc_name: str | None
    financebench_id: str | None
    input_tokens: int
    output_tokens: int
    cost_usd: float | None


class CostTracker:
    def __init__(self, log_path):
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)

    def log(self, pipeline, stage, model, input_tokens, output_tokens, doc_name=None, financebench_id=None):
        prices = PRICING_PER_MILLION_TOKENS.get(model)
        cost_usd = None
        if prices and prices["input"] is not None and prices["output"] is not None:
            cost_usd = (input_tokens * prices["input"] + output_tokens * prices["output"]) / 1_000_000
        record = UsageRecord(
            timestamp=time.time(), pipeline=pipeline, stage=stage, model=model,
            doc_name=doc_name, financebench_id=financebench_id,
            input_tokens=input_tokens, output_tokens=output_tokens, cost_usd=cost_usd,
        )
        with self.log_path.open("a") as f:
            f.write(json.dumps(asdict(record)) + "\n")
        return record


cost_tracker = CostTracker(COSTS_PATH)
print(f"Logging costs to {cost_tracker.log_path}")
print("Note: PRICING_PER_MILLION_TOKENS above is still unfilled (all None), so cost_usd logs as")
print("None for now. Token counts are logged regardless -- fill in published rates before the real")
print("experiment run. Long-context is where this matters most: a whole 10-K/10-Q per query means")
print("every question spends far more input tokens than the other two pipelines' filtered context.")

from groq import Groq

JUDGE_MODEL = "openai/gpt-oss-120b"
judge_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
if judge_client is None:
    print("\nWARNING: GROQ_API_KEY not set -- scoring (Stage 5) will raise on any question that "
          "can't be matched deterministically.")

---
## Stage 1 — Shared utilities

Retry/backoff for flaky API calls, and the generation client — same
`with_retry` design as `vector_rag_pipeline.ipynb`.

In [ ]:
RATE_LIMIT_MARKERS = ("429", "rate limit", "resource_exhausted", "quota")


def _looks_like_rate_limit(exc: Exception) -> bool:
    msg = str(exc).lower()
    return any(marker in msg for marker in RATE_LIMIT_MARKERS)


def with_retry(max_retries: int = 6, base_delay: float = 5.0, max_delay: float = 120.0):
    """Exponential backoff, longer waits for rate-limit-shaped errors.

    Rate-limit errors back off starting at base_delay and double each retry
    (capped at max_delay); any other exception gets one short retry (network
    blip) before propagating, so real bugs still fail fast and loud instead
    of being silently retried into a 6x-slower version of the same crash.
    """
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            delay = base_delay
            last_exc = None
            for attempt in range(max_retries):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:  # noqa: BLE001 - deliberately broad, see docstring
                    last_exc = e
                    if _looks_like_rate_limit(e):
                        print(f"    rate limited ({e!s:.100s}), backing off {delay:.0f}s ...")
                        time.sleep(delay)
                        delay = min(delay * 2, max_delay)
                    elif attempt == 0:
                        print(f"    transient error ({e!s:.100s}), retrying once ...")
                        time.sleep(2.0)
                    else:
                        raise
            raise RuntimeError(f"{fn.__name__} failed after {max_retries} retries") from last_exc
        return wrapper
    return decorator

In [ ]:
from google import genai
import tiktoken

genai_client = genai.Client(api_key=GOOGLE_API_KEY)

GENERATION_MODEL = "gemini-3.5-flash"  # gemini-2.5-flash was deprecated for new API keys/projects
CONTEXT_WINDOW_TOKENS = 1_000_000
CONTEXT_SAFETY_MARGIN = 100_000  # headroom below the hard 1M limit for the prompt wrapper + answer

_approx_encoding = tiktoken.get_encoding("cl100k_base")


def count_tokens_approx(text: str) -> int:
    """Rough token count for the pre-flight "does this fit the context window"
    check. Not billing-exact -- Gemini/DeepSeek don't use this tokenizer --
    real cost comes from each API response's usage_metadata, logged separately
    at generation time. Good enough for a budget guard since 10-Ks are nowhere
    near the 1M-token edge except in rare cases."""
    return len(_approx_encoding.encode(text, disallowed_special=()))


@with_retry()
def _generate(model: str, prompt: str):
    return genai_client.models.generate_content(model=model, contents=prompt)

---
## Stage 2 — Load FinanceBench data

All 150 questions across 84 unique source documents (some documents have multiple questions).

In [ ]:
df_questions = pd.read_json(DATA_DIR / "financebench_open_source.jsonl", lines=True)
df_meta      = pd.read_json(DATA_DIR / "financebench_document_information.jsonl", lines=True)
df           = pd.merge(df_questions, df_meta, on=["doc_name", "company"])

doc_names = sorted(df.doc_name.unique())

print(f"Total questions : {len(df)}")
print(f"Unique documents: {len(doc_names)}")
df[["financebench_id", "doc_name", "question"]].head(3)

---
## Stage 3 — Parse every document's full text

No chunking, no embeddings: parse each PDF into its full text once (page-bounded
so page info is preserved for logging/debugging, even though nothing here
retrieves by page). Saved per document to `DOCS_DIR/{doc_name}.json` —
**resumable**, a document already parsed is skipped. This is the only
preprocessing step in this pipeline (analogous to vector_rag's Stage 4
indexing), and it's pure local PDF parsing — no LLM calls, no cost.

In [ ]:
import pymupdf4llm


def _load_jsonl(path) -> list[dict]:
    if not path.exists():
        return []
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]


def _load_completed_ids(path) -> set:
    return {r["financebench_id"] for r in _load_jsonl(path)}


def _load_stage_records(path) -> dict:
    """financebench_id -> record, for reading a previous stage's output."""
    return {r["financebench_id"]: r for r in _load_jsonl(path)}


def doc_path(docs_dir, doc_name):
    return docs_dir / f"{doc_name}.json"


def is_doc_parsed(docs_dir, doc_name) -> bool:
    return doc_path(docs_dir, doc_name).exists()


def load_doc(docs_dir, doc_name) -> dict:
    return json.loads(doc_path(docs_dir, doc_name).read_text())


def parse_document(pdf_path, doc_name) -> dict:
    """Whole-document parse: PDF -> full markdown text, page-bounded and
    concatenated with a page marker so a human skimming the saved JSON can
    still tell which page any passage came from. pymupdf4llm's page_number is
    1-indexed; stored page_num is 0-indexed to match FinanceBench's
    evidence_page_num convention (the same choice vector_rag's chunker makes)."""
    t0 = time.time()
    md_pages = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)
    pages = [
        {"page_num": p["metadata"]["page_number"] - 1, "text": p["text"]}
        for p in md_pages
    ]
    full_text = "\n\n".join(f"<page {p['page_num']}>\n{p['text']}" for p in pages)
    return {
        "doc_name": doc_name,
        "n_pages": len(pages),
        "token_count_approx": count_tokens_approx(full_text),
        "parse_seconds": time.time() - t0,
        "full_text": full_text,
    }

In [ ]:
def parse_all_documents(doc_names, pdf_dir, docs_dir) -> dict:
    docs_dir.mkdir(parents=True, exist_ok=True)
    errors_log = docs_dir / "parsing_errors.log"
    results = {"parsed": [], "skipped": [], "failed": []}

    for i, doc_name in enumerate(doc_names, start=1):
        if is_doc_parsed(docs_dir, doc_name):
            results["skipped"].append(doc_name)
            print(f"[{i}/{len(doc_names)}] {doc_name}: already parsed, skipping")
            continue

        print(f"[{i}/{len(doc_names)}] {doc_name}: parsing ...")
        try:
            record = parse_document(pdf_dir / f"{doc_name}.pdf", doc_name)
            doc_path(docs_dir, doc_name).write_text(json.dumps(record))
            over = record["token_count_approx"] > (CONTEXT_WINDOW_TOKENS - CONTEXT_SAFETY_MARGIN)
            flag = "  *** EXCEEDS CONTEXT BUDGET ***" if over else ""
            print(f"    {record['n_pages']} pages, ~{record['token_count_approx']:,} tokens, "
                  f"{record['parse_seconds']:.1f}s{flag}")
            results["parsed"].append(doc_name)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{doc_name}: {e}\n")
            results["failed"].append(doc_name)

    print(f"\nParsing summary: {len(results['parsed'])} parsed, {len(results['skipped'])} already done, "
          f"{len(results['failed'])} failed")
    return results

In [ ]:
parse_results = parse_all_documents(doc_names=doc_names, pdf_dir=PDF_DIR, docs_dir=DOCS_DIR)

---
## Stage 4 — Generate the answer directly from the full document

The entire parsed document (Stage 3's `full_text`) goes straight into the
prompt as context — no retrieval, no selection step. If a document's
approximate token count exceeds the context budget
(`CONTEXT_WINDOW_TOKENS - CONTEXT_SAFETY_MARGIN`), the question is skipped
rather than sent anyway and logged to `generation_errors.log`, since a
silently-truncated 10-K would produce a misleadingly-labeled "Failure to
Answer" that actually means "didn't fit."

**Resumable** — needs Stage 3 to have parsed the question's document first.

In [ ]:
GENERATION_PROMPT = """You are a financial analyst answering a question using the complete SEC \
filing below. Answer concisely and precisely, matching the format the question expects (a number, \
a yes/no with brief reasoning, etc). If the filing doesn't contain enough information to answer, \
say so explicitly rather than guessing.

Question: {question}

Filing:
{document_text}

Answer:"""


def generate_answer(question: str, document_text: str, model: str):
    resp = _generate(model, GENERATION_PROMPT.format(question=question, document_text=document_text))
    return resp.text.strip(), resp.usage_metadata


def run_generation_all(df, docs_dir, out_path, generation_model):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    errors_log = out_path.parent / "generation_errors.log"
    completed = _load_completed_ids(out_path)
    doc_cache = {}
    results = {"done": [], "skipped": [], "failed": [], "missing_doc": [], "over_budget": []}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            results["skipped"].append(fb_id)
            continue
        if not is_doc_parsed(docs_dir, row.doc_name):
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- {row.doc_name} not parsed yet")
            results["missing_doc"].append(fb_id)
            continue

        if row.doc_name not in doc_cache:
            doc_cache[row.doc_name] = load_doc(docs_dir, row.doc_name)
        doc = doc_cache[row.doc_name]

        if doc["token_count_approx"] > (CONTEXT_WINDOW_TOKENS - CONTEXT_SAFETY_MARGIN):
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- {row.doc_name} is ~{doc['token_count_approx']:,} "
                  f"tokens, over the context budget")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}: {row.doc_name} over context budget "
                        f"({doc['token_count_approx']:,} tokens)\n")
            results["over_budget"].append(fb_id)
            continue

        print(f"[{i}/{len(df)}] {fb_id}: generating ...")
        try:
            answer, usage = generate_answer(row.question, doc["full_text"], generation_model)
            cost_tracker.log(pipeline="long_context", stage="generation", model=generation_model,
                              input_tokens=usage.prompt_token_count, output_tokens=usage.candidates_token_count,
                              doc_name=row.doc_name, financebench_id=fb_id)
            print(f"    gold : {row.answer[:90]!r}")
            print(f"    model: {answer[:90]!r}")
            record = {
                "financebench_id": fb_id, "doc_name": row.doc_name, "question": row.question,
                "gold_answer": row.answer, "model_answer": answer,
                "generation_model": generation_model, "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
            results["done"].append(fb_id)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}: {e}\n")
            results["failed"].append(fb_id)

    print(f"\nGeneration summary: {len(results['done'])} done, {len(results['skipped'])} already done, "
          f"{len(results['failed'])} failed, {len(results['missing_doc'])} missing parsed doc, "
          f"{len(results['over_budget'])} over context budget")
    return results

In [ ]:
generation_results = run_generation_all(
    df=df, docs_dir=DOCS_DIR, out_path=GENERATION_PATH, generation_model=GENERATION_MODEL,
)

---
## Stage 5 — Score each answer

Same scorer as vector_rag (deterministic numeric matching first, LLM judge
fallback) — copied verbatim from `evaluation/answer_scorer.py`.

**Resumable** — needs Stage 4 to have produced a generated answer for the
question first.

In [ ]:
from typing import Literal

Label = Literal["Correct", "Incorrect", "Failure to Answer"]

_MAGNITUDE_WORDS = {
    "trillion": 1e12, "tn": 1e12, "billion": 1e9, "bn": 1e9,
    "million": 1e6, "mm": 1e6, "thousand": 1e3,
}

# (?<![A-Za-z0-9]) / (?![A-Za-z0-9]) guard against matching digits glued to
# letters -- without them "3M" yields a spurious "3" and "FY2018" yields a
# spurious "2018". A guard on just the single preceding/following character
# (excluding letters only) isn't enough: finditer can still start a fresh match
# *inside* a blocked digit run (e.g. the "0" in "FY2018" is preceded by "2", a
# digit not a letter), so excluding digits too forces the whole contiguous run
# to match-or-skip as one.
#
# The two num alternatives matter: `\d{1,3}(?:,\d{3})+` only matches numbers
# that actually contain comma grouping; `\d+` handles plain digit runs with no
# commas. An earlier version used `(?:,\d{3})*` (zero or more), which let the
# first alternative match just the first 1-3 digits of a longer comma-less
# number and silently truncate it (e.g. "1577.00" -> "157").
_NUM_RE = re.compile(
    r"(?<![A-Za-z0-9])"
    r"(?P<num>-?\$?\s*(?:\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?))"
    r"\s*(?P<mag>trillion|billion|million|thousand|bn|mm|tn|%)?"
    r"(?![A-Za-z0-9])",
    re.IGNORECASE,
)

REL_TOLERANCE = 0.01  # 1% -- covers "small rounding" per the project brief
ABS_TOLERANCE = 0.005  # for gold values near zero, where relative tolerance breaks down

_QUESTION_UNIT_RE = re.compile(r"in\s+(?:usd\s+)?(thousand|million|billion|trillion)s?\b", re.IGNORECASE)


def infer_question_unit_multiplier(question: str) -> float:
    """FinanceBench gold answers are bare numbers already expressed in whatever
    unit the question asks for (e.g. "(in USD millions)"), with no unit word
    repeated in the answer itself. A model's free-text answer usually spells
    the unit back out ("$1,577 million"). This scans the question for that unit
    phrase so both sides normalize to the same base scale. Defaults to 1.0 (no
    scaling) if no unit phrase is found."""
    m = _QUESTION_UNIT_RE.search(question)
    return _MAGNITUDE_WORDS[m.group(1).lower()] if m else 1.0


def extract_numbers(text: str, default_multiplier: float = 1.0) -> list[float]:
    values = []
    for m in _NUM_RE.finditer(text):
        raw = m.group("num").replace("$", "").replace(",", "").strip()
        if not raw or raw in ("-", "."):
            continue
        try:
            value = float(raw)
        except ValueError:
            continue
        mag = m.group("mag")
        if mag == "%":
            multiplier = 1.0
        elif mag:
            multiplier = _MAGNITUDE_WORDS[mag.lower()]
        else:
            multiplier = default_multiplier
        values.append(value * multiplier)
    return values


def numbers_match(a: float, b: float) -> bool:
    if abs(a - b) <= ABS_TOLERANCE:
        return True
    denom = max(abs(a), abs(b))
    return denom > 0 and abs(a - b) / denom <= REL_TOLERANCE


@dataclass
class ScoreResult:
    label: Label
    method: Literal["deterministic", "llm_judge"]
    gold_value: float | None = None
    matched_value: float | None = None
    reasoning: str | None = None


def score_deterministic(question: str, gold_answer: str, model_answer: str) -> ScoreResult | None:
    unit_multiplier = infer_question_unit_multiplier(question)
    gold_numbers = extract_numbers(gold_answer, default_multiplier=unit_multiplier)
    model_numbers = extract_numbers(model_answer, default_multiplier=unit_multiplier)
    if not gold_numbers or not model_numbers:
        return None

    gold_value = gold_numbers[0]  # FinanceBench gold answers are short; first number is the answer
    for candidate in model_numbers:
        if numbers_match(gold_value, candidate):
            return ScoreResult(label="Correct", method="deterministic", gold_value=gold_value, matched_value=candidate)
    return ScoreResult(label="Incorrect", method="deterministic", gold_value=gold_value, matched_value=model_numbers[0])


JUDGE_PROMPT = """You are grading an AI system's answer to a question about a company's SEC filing, \
against a human-verified gold answer. Classify the model's answer into exactly one of three categories:

- "Correct": the model's answer matches the gold answer in substance (numbers may be phrased \
differently -- e.g. rounding, units -- but must be materially the same value or conclusion).
- "Incorrect": the model gave a definite answer, but it's wrong.
- "Failure to Answer": the model declined to answer, said the information wasn't available/sufficient, \
or gave a non-answer, rather than committing to a (possibly wrong) answer.

Question: {question}

Gold answer: {gold_answer}

Model answer: {model_answer}

Respond with ONLY a JSON object, no other text: {{"label": "<one of the three categories above>", "reasoning": "<one sentence>"}}"""


def score_with_judge(question, gold_answer, model_answer, client, model="openai/gpt-oss-120b"):
    resp = client.chat.completions.create(
        model=model, max_tokens=300,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(question=question, gold_answer=gold_answer, model_answer=model_answer)}],
    )
    raw = resp.choices[0].message.content.strip()
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    parsed = json.loads(match.group(0) if match else raw)

    label = parsed["label"]
    if label not in ("Correct", "Incorrect", "Failure to Answer"):
        raise ValueError(f"Judge returned an unrecognized label: {label!r}")

    result = ScoreResult(label=label, method="llm_judge", reasoning=parsed.get("reasoning"))
    usage = {"input_tokens": resp.usage.prompt_tokens, "output_tokens": resp.usage.completion_tokens}
    return result, usage


def score_answer(question, gold_answer, model_answer, judge_client=None, judge_model="openai/gpt-oss-120b"):
    result = score_deterministic(question, gold_answer, model_answer)
    if result is not None:
        return result, None
    if judge_client is None:
        raise ValueError(
            "Deterministic matching failed and no judge_client was provided to fall back to the LLM judge."
        )
    return score_with_judge(question, gold_answer, model_answer, judge_client, judge_model)

In [ ]:
def run_scoring_all(df, generation_path, out_path, judge_client, judge_model):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    errors_log = out_path.parent / "scoring_errors.log"
    generation_records = _load_stage_records(generation_path)
    completed = _load_completed_ids(out_path)
    results = {"done": [], "skipped": [], "failed": [], "missing_generation": []}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            results["skipped"].append(fb_id)
            continue
        if fb_id not in generation_records:
            results["missing_generation"].append(fb_id)
            continue

        print(f"[{i}/{len(df)}] {fb_id}: scoring ...")
        try:
            gen = generation_records[fb_id]
            score_result, usage = score_answer(row.question, row.answer, gen["model_answer"],
                                                judge_client=judge_client, judge_model=judge_model)
            if usage is not None:
                cost_tracker.log(pipeline="long_context", stage="judge", model=judge_model,
                                  input_tokens=usage["input_tokens"], output_tokens=usage["output_tokens"],
                                  doc_name=row.doc_name, financebench_id=fb_id)
            if score_result.reasoning:
                print(f"    reasoning: {score_result.reasoning[:110]!r}")
            record = {
                "financebench_id": fb_id, "doc_name": row.doc_name,
                "score_label": score_result.label, "score_method": score_result.method,
                "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
            print(f"    done -- {score_result.label} ({score_result.method})")
            results["done"].append(fb_id)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}: {e}\n")
            results["failed"].append(fb_id)

    print(f"\nScoring summary: {len(results['done'])} done, {len(results['skipped'])} already done, "
          f"{len(results['failed'])} failed, {len(results['missing_generation'])} missing generated answer")
    return results


scoring_results = run_scoring_all(
    df=df, generation_path=GENERATION_PATH, out_path=SCORING_PATH,
    judge_client=judge_client, judge_model=JUDGE_MODEL,
)

---
## Stage 6 — Summarize

Answer-quality breakdown and token totals per stage, computed over whatever's
in Stages 4-5's output files so far — doesn't require all 150 to be done. No
retrieval-quality section (not applicable, see the top of this notebook) and
no latency number here — see Stage 7.

In [ ]:
from collections import Counter, defaultdict


def summarize_results(scoring_path, cost_log_path=None):
    scoring = _load_jsonl(scoring_path)
    if not scoring:
        print(f"No scored results yet in {scoring_path}")
        return {}

    n = len(scoring)
    label_counts = Counter(r["score_label"] for r in scoring)
    summary = {
        "n_questions": n,
        "answer_quality": {label: {"count": c, "pct": round(100 * c / n, 1)} for label, c in label_counts.items()},
    }

    if cost_log_path is not None:
        cost_records = _load_jsonl(cost_log_path)
        by_stage = defaultdict(lambda: {"input_tokens": 0, "output_tokens": 0, "calls": 0, "cost_usd": 0.0, "cost_unpriced_calls": 0})
        for r in cost_records:
            s = by_stage[r["stage"]]
            s["input_tokens"] += r["input_tokens"]
            s["output_tokens"] += r["output_tokens"]
            s["calls"] += 1
            if r["cost_usd"] is not None:
                s["cost_usd"] += r["cost_usd"]
            else:
                s["cost_unpriced_calls"] += 1
        summary["tokens_by_stage"] = dict(by_stage)
        summary["total_cost_usd"] = sum(s["cost_usd"] for s in by_stage.values())
        summary["total_unpriced_calls"] = sum(s["cost_unpriced_calls"] for s in by_stage.values())

    return summary


def print_summary(summary: dict) -> None:
    if not summary:
        return
    n = summary["n_questions"]
    print(f"Questions scored: {n}\n")
    print("Answer quality:")
    for label, d in summary["answer_quality"].items():
        print(f"  {label:<20} {d['count']:>4}  ({d['pct']}%)")

    if "tokens_by_stage" in summary:
        print("\nTokens by stage (input tokens dominated by generation here -- the whole filing is "
              "the prompt on every question, unlike vector/vectorless RAG's filtered context):")
        for stage, s in summary["tokens_by_stage"].items():
            print(f"  {stage:<24} calls={s['calls']:<5} in={s['input_tokens']:<10} out={s['output_tokens']}")
        note = "" if summary["total_unpriced_calls"] == 0 else (
            f" ({summary['total_unpriced_calls']} calls unpriced -- fill in PRICING_PER_MILLION_TOKENS)")
        print(f"\nDollar cost: ${summary['total_cost_usd']:.4f} total{note}")

    print("\nNote: retrieval quality (Recall@k/MRR@k) does not apply to this pipeline -- no "
          "retrieval step exists to evaluate (see CLAUDE.md).")
    print("Note: per-question end-to-end latency is not measured here either -- see the dedicated "
          "timing pass (Stage 7) below.")


summary = summarize_results(SCORING_PATH, cost_log_path=COSTS_PATH)
print_summary(summary)

### Next steps
1. Check `DOCS_DIR/parsing_errors.log`, `generation_errors.log` (also lists any
   question skipped for exceeding the context budget), and `scoring_errors.log`
   for anything that needs a second look.
2. Fill in `PRICING_PER_MILLION_TOKENS` in Stage 0 with current published
   rates. This matters more here than in the other two pipelines: every
   question's input is a full 10-K/10-Q, so per-query cost is expected to
   dominate the accuracy-latency-cost frontier for this architecture.
3. Once this is stable, wire in DeepSeek V4 as a second `generation_model` and
   re-run Stage 4 with it — Stage 3's parsed documents are model-independent
   and don't need to be redone.
4. Run Stage 7 below for the median-of-N latency number the project brief
   asks for.

---
## Stage 7 — Dedicated latency timing pass

Stage 4 is already resumable, but it doesn't time itself or repeat each
question for a median. This stage is a small, dedicated, **strictly
sequential** pass over a fixed sample, run **fully fresh** each time (context
assembly + generate, timed end-to-end and per sub-stage), matching the
project brief's "run each query multiple times, report the median."

Document parsing (Stage 3) is excluded — same reasoning as vector_rag's Stage
12: that's one-time preprocessing per document, not something a live query
pays for. Scoring is excluded too — grading isn't part of response time.

Default: **10 questions × 3 repeats = 30 timed passes**.

In [ ]:
import statistics

LATENCY_SAMPLE_SIZE = 10   # questions
LATENCY_REPEATS = 3        # per project brief: run each query multiple times, report the median


def time_single_pass(row, doc, generation_model):
    """One timed pass: context assembly (reading the already-parsed document
    text -- cheap disk I/O, what a live system would keep warm in memory) +
    generate. Returns (timings_sec dict, model_answer, usage)."""
    timings = {}
    t_total0 = time.time()

    t0 = time.time()
    document_text = doc["full_text"]
    timings["context_assembly"] = time.time() - t0

    t0 = time.time()
    answer, usage = generate_answer(row.question, document_text, generation_model)
    timings["generate"] = time.time() - t0

    timings["total"] = time.time() - t_total0
    return timings, answer, usage


def run_latency_pass(df, docs_dir, generation_model, out_path,
                      sample_size=LATENCY_SAMPLE_SIZE, repeats=LATENCY_REPEATS, seed=42):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    parseable = df[df.doc_name.apply(lambda d: is_doc_parsed(docs_dir, d))]
    parseable = parseable[parseable.doc_name.apply(
        lambda d: load_doc(docs_dir, d)["token_count_approx"] <= (CONTEXT_WINDOW_TOKENS - CONTEXT_SAFETY_MARGIN)
    )]
    if len(parseable) < len(df):
        print(f"Note: {len(df) - len(parseable)} questions excluded from the sampling pool "
              f"(document not parsed yet, or over the context budget)")
    sample = parseable.sample(n=min(sample_size, len(parseable)), random_state=seed)

    completed_pairs = {(r["financebench_id"], r["repeat"]) for r in _load_jsonl(out_path)}
    total_runs = len(sample) * repeats
    print(f"Timing {len(sample)} questions x {repeats} repeats = {total_runs} passes "
          f"({len(completed_pairs)} already done)")

    doc_cache = {}
    for qi, row in enumerate(sample.itertuples(), start=1):
        if row.doc_name not in doc_cache:
            doc_cache[row.doc_name] = load_doc(docs_dir, row.doc_name)
        doc = doc_cache[row.doc_name]

        for r in range(1, repeats + 1):
            if (row.financebench_id, r) in completed_pairs:
                continue
            print(f"[{qi}/{len(sample)} x {r}/{repeats}] {row.financebench_id} ({row.doc_name}) ...")
            try:
                timings, answer, usage = time_single_pass(row, doc, generation_model)
                cost_tracker.log(pipeline="long_context", stage="generation_latency_pass", model=generation_model,
                                  input_tokens=usage.prompt_token_count, output_tokens=usage.candidates_token_count,
                                  doc_name=row.doc_name, financebench_id=row.financebench_id)
                print(f"    total: {timings['total']:.2f}s  "
                      f"(context={timings['context_assembly']:.2f} generate={timings['generate']:.2f})")
                record = {
                    "financebench_id": row.financebench_id, "doc_name": row.doc_name, "repeat": r,
                    "timings_sec": timings, "timestamp": time.time(),
                }
                with out_path.open("a") as f:
                    f.write(json.dumps(record) + "\n")
            except Exception as e:
                print(f"    FAILED: {e}")

In [ ]:
run_latency_pass(
    df=df, docs_dir=DOCS_DIR, generation_model=GENERATION_MODEL, out_path=LATENCY_PATH,
)

In [ ]:
def summarize_latency(out_path):
    """Median-of-N per question (per the project brief), then median across
    questions for the headline number -- robust to one slow or retried pass
    dragging the number the way a mean would."""
    records = _load_jsonl(out_path)
    if not records:
        print(f"No latency data yet in {out_path}")
        return {}

    by_question = defaultdict(list)
    for r in records:
        by_question[r["financebench_id"]].append(r["timings_sec"]["total"])
    per_question_median = {fb_id: statistics.median(vals) for fb_id, vals in by_question.items()}
    overall_median = statistics.median(per_question_median.values())

    stage_names = [k for k in records[0]["timings_sec"] if k != "total"]
    stage_medians = {stage: statistics.median(r["timings_sec"][stage] for r in records) for stage in stage_names}

    return {
        "n_questions": len(by_question), "n_runs": len(records),
        "per_question_median_sec": per_question_median,
        "overall_median_sec": overall_median,
        "stage_medians_sec": stage_medians,
    }


def print_latency_summary(summary: dict) -> None:
    if not summary:
        return
    print(f"Latency sample: {summary['n_questions']} questions, {summary['n_runs']} total timed passes")
    print(f"\nMedian end-to-end latency (median-of-N per question, then median across questions): "
          f"{summary['overall_median_sec']:.2f}s")
    print("\nMedian latency by stage (across all timed passes):")
    for stage, m in summary["stage_medians_sec"].items():
        print(f"  {stage:<18} {m:.2f}s")


latency_summary = summarize_latency(LATENCY_PATH)
print_latency_summary(latency_summary)